#### **Load and inspect**

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/2026_Data_IPL.csv')
print(df.shape)
df.head()

(17527, 16)


,Match_ID,Match_Name,Date,Innings,Batting_Team,Bowling_Team,Batter,Bowler,Runs,Total_Balls,Extras,Boundary,Overs,Wickets,Total_Runs,Wining Team
0,1,RCB VS SRH,28-Mar-26,1st,SRH,RCB,Travis Head,Jacob Duffy,0.0,1.0,0,0.0,0.1,0.0,0.0,0
1,1,RCB VS SRH,28-Mar-26,1st,SRH,RCB,Travis Head,Jacob Duffy,1.0,2.0,0,0.0,0.2,0.0,1.0,0
2,1,RCB VS SRH,28-Mar-26,1st,SRH,RCB,Abhishek Sharma,Jacob Duffy,0.0,3.0,0,0.0,0.3,0.0,0.0,0
3,1,RCB VS SRH,28-Mar-26,1st,SRH,RCB,Abhishek Sharma,Jacob Duffy,6.0,4.0,0,6.0,0.4,0.0,7.0,0
4,1,RCB VS SRH,28-Mar-26,1st,SRH,RCB,Abhishek Sharma,Jacob Duffy,0.0,5.0,0,0.0,0.5,0.0,7.0,0


#### **Clean column names (strip trailing spaces)**

In [2]:
df.columns = df.columns.str.strip()
print(df.columns.tolist())

['Match_ID', 'Match_Name', 'Date', 'Innings', 'Batting_Team', 'Bowling_Team', 'Batter', 'Bowler', 'Runs', 'Total_Balls', 'Extras', 'Boundary', 'Overs', 'Wickets', 'Total_Runs', 'Wining Team']


#### **Strip whitespace from all string/object columns**

In [3]:
str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].str.strip()

# Verify the fix worked
print(sorted(df['Batting_Team'].unique()))
print(sorted(df['Match_Name'].unique())[:5])

['CSK', 'DC', 'GT', 'KKR', 'LSG', 'MI', 'PBKS', 'RCB', 'RR', 'SRH']
['CSK VS DC', 'CSK VS KKR', 'CSK VS LSG', 'CSK VS MI', 'CSK VS PBKS']


C:\Users\phamd\AppData\Local\Temp\ipykernel_8896\3937718151.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include='object').columns


#### **Drop the null row and parse dates**

In [4]:
df = df.dropna(subset=['Bowler', 'Runs', 'Total_Balls'])  # drops the 1 incomplete row

df['Date'] = pd.to_datetime(df['Date'], format='%d-%b-%y')
print(df.shape)
print(df['Date'].min(), df['Date'].max())

(17526, 16)
2026-03-28 00:00:00 2026-05-31 00:00:00


#### **Handle the messy Wining Team column**

In [5]:
# '0' = likely result not recorded / placeholder, 'ABONDAND' = abandoned match
print(df['Wining Team'].value_counts())

# Flag match status for later filtering
df['Match_Status'] = np.where(df['Wining Team'] == 'ABONDAND', 'Abandoned',
                        np.where(df['Wining Team'] == '0', 'No Result', 'Completed'))

print(df['Match_Status'].value_counts())

Wining Team
0       17453
RCB        11
GT         10
RR          9
SRH         9
PBKS        7
DC          7
CSK         6
KKR         6
MI          4
LSG         4
Name: count, dtype: int64
Match_Status
No Result    17453
Completed       73
Name: count, dtype: int64


#### **Build the Matches table**

In [6]:
matches = df.groupby('Match_ID').agg(
    Match_Name=('Match_Name', 'first'),
    Date=('Date', 'first'),
    Season=('Date', lambda x: 2026),
    Team1=('Batting_Team', 'first'),
    Winning_Team=('Wining Team', 'last'),   # <-- was 'first', now 'last'
    Match_Status=('Wining Team', lambda x: 'Abandoned' if x.iloc[-1] == 'ABONDAND'
                                   else ('No Result' if x.iloc[-1] == '0' else 'Completed'))
).reset_index()

print(matches['Match_Status'].value_counts())
print(matches['Winning_Team'].value_counts())
matches.head()

Match_Status
Completed    73
No Result     2
Name: count, dtype: int64
Winning_Team
RCB     11
GT      10
RR       9
SRH      9
PBKS     7
DC       7
CSK      6
KKR      6
LSG      4
MI       4
0        2
Name: count, dtype: int64


,Match_ID,Match_Name,Date,Season,Team1,Winning_Team,Match_Status
0,1,RCB VS SRH,2026-03-28,2026,SRH,RCB,Completed
1,10,SRH VS LSG,2026-04-05,2026,SRH,LSG,Completed
2,11,RCB VS CSK,2026-04-05,2026,RCB,RCB,Completed
3,12,KKR VS PBKS,2026-04-06,2026,KKR,0,No Result
4,13,RR VS MI,2026-04-07,2026,RR,RR,Completed


#### **Build match-level total runs (for the time series chart)**

In [7]:
# Total runs per team per match (using max cumulative Total_Runs per innings)
runs_per_match = df.groupby(['Match_ID', 'Date', 'Batting_Team'])['Total_Runs'].max().reset_index()
runs_per_match = runs_per_match.rename(columns={'Batting_Team': 'Team', 'Total_Runs': 'Runs_Scored'})

print(runs_per_match.shape)
runs_per_match.head()

(144, 4)


,Match_ID,Date,Team,Runs_Scored
0,1,2026-03-28,RCB,203.0
1,1,2026-03-28,SRH,201.0
2,10,2026-04-05,LSG,160.0
3,10,2026-04-05,SRH,156.0
4,11,2026-04-05,CSK,207.0


#### **Save processed data**

In [8]:
df.to_csv('../data/processed/ball_by_ball_clean.csv', index=False)
matches.to_csv('../data/processed/matches.csv', index=False)
runs_per_match.to_csv('../data/processed/runs_per_match.csv', index=False)

#### **Compute per-ball wicket events**

In [9]:
df = df.sort_values(['Match_ID', 'Innings', 'Total_Balls'])

# Wicket fell on this ball if cumulative Wickets increased vs previous ball in same innings
df['Wicket_Event'] = df.groupby(['Match_ID', 'Innings'])['Wickets'].diff().fillna(df['Wickets']).gt(0).astype(int)

print(df['Wicket_Event'].sum(), "total wicket events")

983 total wicket events


#### **Top 10 run-scorers**

In [10]:
top_batters = df.groupby('Batter')['Runs'].sum().sort_values(ascending=False).head(10).reset_index()
top_batters.columns = ['Batter', 'Total_Runs']
top_batters

,Batter,Total_Runs
0,Vaibhav Suryavanshi,799.0
1,Shubman Gill,772.0
2,Sai Sudharsan,762.0
3,Virat Kohli,739.0
4,Henrich Klaasen,651.0
5,KL Rahul,619.0
6,Ishan Kisan,616.0
7,Abhishek Sharma,605.0
8,Mitchell Marsh,586.0
9,Prabhsimran Singh,535.0


#### **Top 10 wicket-takers**

In [11]:
top_bowlers = df.groupby('Bowler')['Wicket_Event'].sum().sort_values(ascending=False).head(10).reset_index()
top_bowlers.columns = ['Bowler', 'Total_Wickets']
top_bowlers

,Bowler,Total_Wickets
0,Bhuvneshvar Kumar,37
1,Prince Yadav,33
2,Jofra Archer,29
3,Kagiso Rabada,27
4,Eshan Malinga,26
5,Arshdeep Singh,25
6,Shahbaz Ahmed,22
7,Anshul Kamboj,21
8,Mayank Yadav,21
9,Rashid Khan,20


#### **Team win percentages**

In [12]:
completed = matches[matches['Match_Status'] == 'Completed']

# Count matches played per team (as Team1 in our matches table only captures batting-first team,
# so we need total appearances from the ball-by-ball data instead)
teams_in_matches = df.groupby('Match_ID')['Batting_Team'].unique().reset_index()
teams_in_matches['Team_A'] = teams_in_matches['Batting_Team'].apply(lambda x: x[0])
teams_in_matches['Team_B'] = teams_in_matches['Batting_Team'].apply(lambda x: x[1] if len(x) > 1 else None)

played_long = pd.concat([
    teams_in_matches[['Match_ID', 'Team_A']].rename(columns={'Team_A': 'Team'}),
    teams_in_matches[['Match_ID', 'Team_B']].rename(columns={'Team_B': 'Team'})
])

matches_played = played_long.merge(matches[['Match_ID', 'Winning_Team', 'Match_Status']], on='Match_ID')
matches_played = matches_played[matches_played['Match_Status'] == 'Completed']

wins = matches_played[matches_played['Team'] == matches_played['Winning_Team']].groupby('Team').size()
total = matches_played.groupby('Team').size()

win_pct = (wins / total * 100).round(1).reset_index()
win_pct.columns = ['Team', 'Win_Percentage']
win_pct = win_pct.sort_values('Win_Percentage', ascending=False)
win_pct

,Team,Win_Percentage
7,RCB,68.8
2,GT,60.0
9,SRH,60.0
6,PBKS,58.3
8,RR,56.2
1,DC,46.2
3,KKR,46.2
0,CSK,42.9
5,MI,28.6
4,LSG,23.1


#### **Runs per match (time series)**

In [13]:
match_totals = runs_per_match.groupby(['Match_ID', 'Date'])['Runs_Scored'].sum().reset_index()
match_totals = match_totals.sort_values('Date')
match_totals

,Match_ID,Date,Runs_Scored
0,1,2026-03-28,404.0
11,2,2026-03-29,444.0
22,3,2026-03-30,255.0
33,4,2026-03-31,327.0
44,5,2026-04-01,286.0
...,...,...,...
67,70,2026-05-24,366.0
68,71,2026-05-26,416.0
69,72,2026-05-27,439.0
70,73,2026-05-29,219.0


#### **Chart 1: Runs per match**

In [14]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [15]:
import plotly.graph_objects as go

teams = sorted(runs_per_match['Team'].unique())

fig1 = go.Figure()

# Add one trace per team (all visible by default, dropdown toggles which show)
for team in teams:
    team_data = runs_per_match[runs_per_match['Team'] == team].sort_values('Date')
    fig1.add_trace(go.Scatter(
        x=team_data['Date'], y=team_data['Runs_Scored'],
        mode='lines+markers', name=team, visible=True
    ))

# Dropdown: "All Teams" vs individual team
buttons = [dict(label='All Teams', method='update',
                 args=[{'visible': [True]*len(teams)}])]
for i, team in enumerate(teams):
    visibility = [False]*len(teams)
    visibility[i] = True
    buttons.append(dict(label=team, method='update', args=[{'visible': visibility}]))

fig1.update_layout(
    title='Runs per Match (2026 Season)',
    xaxis_title='Match Date', yaxis_title='Total Runs',
    updatemenus=[dict(buttons=buttons, direction='down', x=1.15, y=1.15)]
)
fig1.show()

#### **Chart 2: Top 10 run-scorers**

In [16]:
# Recompute per team
batters_by_team = df.groupby(['Batting_Team', 'Batter'])['Runs'].sum().reset_index()

def top10_batters(team=None):
    d = batters_by_team if team is None else batters_by_team[batters_by_team['Batting_Team'] == team]
    return d.groupby('Batter')['Runs'].sum().sort_values(ascending=False).head(10)

teams = sorted(df['Batting_Team'].unique())

fig2 = go.Figure()
# Trace 0 = All teams (default visible)
all_top = top10_batters().sort_values()
fig2.add_trace(go.Bar(x=all_top.values, y=all_top.index, orientation='h', visible=True, name='All'))

# One trace per team (hidden by default)
team_traces = []
for team in teams:
    t = top10_batters(team).sort_values()
    fig2.add_trace(go.Bar(x=t.values, y=t.index, orientation='h', visible=False, name=team))
    team_traces.append(team)

buttons = [dict(label='All Teams', method='update',
                 args=[{'visible': [True] + [False]*len(teams)}])]
for i, team in enumerate(teams):
    vis = [False]*(len(teams)+1)
    vis[i+1] = True
    buttons.append(dict(label=team, method='update', args=[{'visible': vis}]))

fig2.update_layout(
    title='Top 10 Run-Scorers',
    xaxis_title='Total Runs', yaxis_title='',
    updatemenus=[dict(buttons=buttons, direction='down', x=1.15, y=1.15)]
)
fig2.show()

#### **Chart 3: Top 10 wicket-takers**

In [17]:
bowlers_by_team = df.groupby(['Bowling_Team', 'Bowler'])['Wicket_Event'].sum().reset_index()

def top10_bowlers(team=None):
    d = bowlers_by_team if team is None else bowlers_by_team[bowlers_by_team['Bowling_Team'] == team]
    return d.groupby('Bowler')['Wicket_Event'].sum().sort_values(ascending=False).head(10)

fig3 = go.Figure()
all_top_b = top10_bowlers().sort_values()
fig3.add_trace(go.Bar(x=all_top_b.values, y=all_top_b.index, orientation='h', visible=True, name='All'))

for team in teams:
    t = top10_bowlers(team).sort_values()
    fig3.add_trace(go.Bar(x=t.values, y=t.index, orientation='h', visible=False, name=team))

buttons = [dict(label='All Teams', method='update',
                 args=[{'visible': [True] + [False]*len(teams)}])]
for i, team in enumerate(teams):
    vis = [False]*(len(teams)+1)
    vis[i+1] = True
    buttons.append(dict(label=team, method='update', args=[{'visible': vis}]))

fig3.update_layout(
    title='Top 10 Wicket-Takers',
    xaxis_title='Total Wickets', yaxis_title='',
    updatemenus=[dict(buttons=buttons, direction='down', x=1.15, y=1.15)]
)
fig3.show()

#### **Chart 4: Team win percentages**

In [18]:
# Win % is already one bar per team, so "filter" here highlights the selected team
# instead of hiding others (hiding would leave an empty chart for single-team view)
colors_default = ['steelblue'] * len(win_pct)

fig4 = go.Figure(go.Bar(x=win_pct['Team'], y=win_pct['Win_Percentage'],
                          marker_color=colors_default))

buttons = [dict(label='All Teams', method='update',
                 args=[{'marker.color': [colors_default]}])]
for team in win_pct['Team']:
    colors = ['crimson' if t == team else 'lightgray' for t in win_pct['Team']]
    buttons.append(dict(label=team, method='update', args=[{'marker.color': [colors]}]))

fig4.update_layout(
    title='Team Win Percentage (2026 Season)',
    xaxis_title='Team', yaxis_title='Win %',
    updatemenus=[dict(buttons=buttons, direction='down', x=1.15, y=1.15)]
)
fig4.show()

#### **Export each chart as PNG**

In [19]:
import os
os.makedirs('../exports/charts', exist_ok=True)

fig1.write_image('../exports/charts/01_runs_per_match.png', width=1200, height=600, scale=2)
fig2.write_image('../exports/charts/02_top_run_scorers.png', width=1200, height=600, scale=2)
fig3.write_image('../exports/charts/03_top_wicket_takers.png', width=1200, height=600, scale=2)
fig4.write_image('../exports/charts/04_team_win_pct.png', width=1200, height=600, scale=2)

print("PNGs exported to exports/charts/")

PNGs exported to exports/charts/


In [20]:
from datetime import date

figs = [fig1, fig2, fig3, fig4]
titles = ["Runs per Match", "Top 10 Run-Scorers", "Top 10 Wicket-Takers", "Team Win Percentage"]

html_parts = [f"""
<html>
<head><title>IPL 2026 Dashboard</title></head>
<body style="font-family:Arial, sans-serif; margin:40px;">
<h1>IPL 2026 Statistics Dashboard</h1>
<p>Data source: Kaggle - IPL 2026 Ball-by-Ball Dataset (extracted 9/16/2026)</p>
"""]

for i, (fig, title) in enumerate(zip(figs, titles)):
    include_js = 'cdn' if i == 0 else False  # only load plotly.js once
    html_parts.append(f"<h2>{title}</h2>")
    html_parts.append(fig.to_html(full_html=False, include_plotlyjs=include_js))

html_parts.append("</body></html>")

with open('../exports/ipl_dashboard.html', 'w', encoding='utf-8') as f:
    f.write("".join(html_parts))

print("Dashboard saved to ../exports/ipl_dashboard.html")

Dashboard saved to ../exports/ipl_dashboard.html
